<a href="https://colab.research.google.com/github/HugoCortazar/Mineria-de-Textos-/blob/mejora-dataset-biologia/MTPracticaFinal_intermedia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Minería de Textos · Textos científicos en Esperanto

**Entrega intermedia — Fases 1 y 2**

El objetivo del proyecto es clasificar párrafos en Esperanto según pertenezcan a **biología (`biologio`)**, **química (`kemio`)** o **física (`fiziko`)**. Elegimos estas categorías porque forman parte del ámbito científico y comparten algunos términos, aunque cada una tiene también vocabulario propio.

En esta entrega preparamos un dataset a partir de Wikipedia en Esperanto y desarrollamos un procesador lingüístico sencillo basado en reglas. El entrenamiento y la evaluación del clasificador se realizarán en la fase final.

## 1. Construcción del dataset

### 1.1. Fuente y configuración

Recuperamos artículos de la Wikipedia en Esperanto mediante su API. Para cada temática seleccionamos varios artículos y utilizamos sus párrafos como ejemplos de entrenamiento.

El parámetro `DESCARGAR_DESDE_WIKIPEDIA` permite generar el CSV o cargar uno ya existente. Guardamos únicamente las columnas exigidas por el enunciado: `text` y `class`.

In [1]:
# Configuración general y artículos seleccionados
import json
import re
import time
import urllib.parse
import urllib.request

from urllib.error import HTTPError, URLError
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime

import pandas as pd
from IPython.display import display

# True: consultar Wikipedia; False: usar el CSV que ya se ha generado.
DESCARGAR_DESDE_WIKIPEDIA = True

GITHUB_CSV_URL = (
    "https://raw.githubusercontent.com/"
    "HugoCortazar/Mineria-de-Textos-/main/"
    "esperanto_scienco_dataset.csv"
)

WIKIPEDIA_API_URL = "https://eo.wikipedia.org/w/api.php"

USER_AGENT = (
    "MineriaTextosEsperanto/1.0 "
    "(https://github.com/HugoCortazar/Mineria-de-Textos-/)"
)

LONGITUD_MINIMA_PARRAFO = 60
PAUSA_ENTRE_ARTICULOS = 2

# Evitamos guardar el CSV si alguna clase tiene muy pocos ejemplos.
# La recomendación del profesor es superar los 300 párrafos por clase.
MIN_PARRAFOS_POR_CLASE = 30

CATEGORIAS = {
    "biologio": [
        "Biologio", "Ĉelo", "Genetiko", "Evoluismo",
        "Fotosintezo", "Zoologio", "Botaniko",
        "Mikrobiologio", "Ekologio", "DNA",
        "Anatomio", "Biodiverseco", "Metabolo"
    ],
    "kemio": [
        "Kemio", "Atomo", "Perioda tabelo", "Kemia ligo",
        "Organika kemio", "Neorganika kemio",
        "Kemia reakcio", "Acido", "Molekulo", "Katalizilo"
    ],
    "fiziko": [
        "Fiziko", "Klasika mekaniko", "Termodinamiko",
        "Elektromagnetismo", "Kvantuma mekaniko",
        "Relativeco", "Optiko", "Kinetika energio",
        "Partikla fiziko", "Astrofiziko", "Gravito",
        "Akustiko", "Nuklea fiziko"
    ]
}

SECCIONES_RUIDOSAS = {
    "referencoj",
    "referencoj kaj notoj",
    "eksteraj ligiloj",
    "vidu ankaŭ",
    "vidu ankau",
    "bibliografio",
    "literaturo",
    "notoj",
    "fontoj"
}

print("Configuración cargada.")
print("Descarga desde Wikipedia:", DESCARGAR_DESDE_WIKIPEDIA)
print("Categorías:", list(CATEGORIAS.keys()))


Configuración cargada.
Descarga desde Wikipedia: True
Categorías: ['biologio', 'kemio', 'fiziko']


### 1.2. Descarga de artículos

Las peticiones se hacen de forma individual y con una pausa entre artículos para no sobrecargar la API. Si Wikipedia devuelve un error temporal, el código intenta repetir la petición. Esto facilita volver a generar el dataset.

In [2]:
# Lectura de artículos desde la API de Wikipedia
def calcular_espera_reintento(cabeceras, intento):
    """Respeta Retry-After; si falta, utiliza una pausa creciente."""

    espera_base = 10 * (2 ** intento)
    retry_after = cabeceras.get("Retry-After") if cabeceras else None

    if not retry_after:
        return espera_base

    try:
        return max(espera_base, float(retry_after))
    except (TypeError, ValueError):
        try:
            fecha = parsedate_to_datetime(retry_after)

            if fecha.tzinfo is None:
                fecha = fecha.replace(tzinfo=timezone.utc)

            segundos = (
                fecha - datetime.now(timezone.utc)
            ).total_seconds()

            return max(espera_base, segundos, 0)

        except (TypeError, ValueError, OverflowError):
            return espera_base


def obtener_texto_wikipedia(titulo, max_intentos=4):
    """
    Devuelve el texto plano de un único artículo de Wikipedia en Esperanto.
    """

    parametros = {
        "action": "query",
        "prop": "extracts",
        "explaintext": 1,
        "redirects": 1,
        "titles": titulo,
        "format": "json",
        "formatversion": 2
    }

    url = (
        WIKIPEDIA_API_URL
        + "?"
        + urllib.parse.urlencode(parametros)
    )

    for intento in range(max_intentos):

        solicitud = urllib.request.Request(
            url,
            headers={
                "User-Agent": USER_AGENT,
                "Accept": "application/json"
            }
        )

        try:
            with urllib.request.urlopen(
                solicitud, timeout=30
            ) as respuesta:

                datos = json.loads(
                    respuesta.read().decode("utf-8")
                )

            paginas = datos.get("query", {}).get("pages", [])

            if not paginas:
                print(
                    f"Aviso: la API no devolvió páginas para '{titulo}'."
                )
                return ""

            pagina = paginas[0]

            if pagina.get("missing"):
                print(
                    f"Aviso: no existe el artículo '{titulo}' "
                    "en Wikipedia Esperanto."
                )
                return ""

            texto = pagina.get("extract", "")

            if not texto.strip():
                print(
                    f"Aviso: el artículo '{titulo}' "
                    "no contiene extracto de texto."
                )
                return ""

            return texto

        except HTTPError as error:

            if error.code not in (429, 503):
                print(
                    f"Aviso HTTP {error.code} para '{titulo}': "
                    f"{error.reason}"
                )
                return ""

            if intento == max_intentos - 1:
                raise RuntimeError(
                    f"Wikipedia continúa limitando las peticiones "
                    f"para '{titulo}' tras {max_intentos} intentos. "
                    "Espera antes de volver a ejecutar."
                ) from error

            espera = calcular_espera_reintento(
                error.headers, intento
            )

            # Evitamos esperas excesivas si Wikipedia limita las peticiones.
            if espera > 300:
                raise RuntimeError(
                    f"Wikipedia indica que hay que esperar "
                    f"{espera:.0f} segundos antes de reintentar "
                    f"'{titulo}'. Ejecuta el cuaderno más tarde."
                ) from error

            print(
                f"HTTP {error.code} al consultar '{titulo}'. "
                f"Esperando {espera:.0f} segundos antes de reintentar."
            )

            time.sleep(espera)

        except (URLError, TimeoutError) as error:

            if intento == max_intentos - 1:
                raise RuntimeError(
                    f"No se pudo conectar con Wikipedia para "
                    f"'{titulo}' después de {max_intentos} intentos."
                ) from error

            espera = 5 * (2 ** intento)

            print(
                f"Error de conexión para '{titulo}'. "
                f"Reintento en {espera} segundos."
            )

            time.sleep(espera)

    return ""


### 1.3. Limpieza de párrafos

Eliminamos apartados de referencias, enlaces y elementos de formato. También descartamos los fragmentos de menos de 60 caracteres, que suelen aportar poco contexto. Sustituimos el carácter `;` porque el CSV utiliza ese separador.

No todos los bloques descartados son necesariamente *ruido temático*: esta limpieza solo controla la forma y la longitud del texto.

In [3]:
# Limpieza y filtrado de fragmentos
def limpiar_y_filtrar_parrafos(texto_articulo, clase):
    """
    Devuelve:
    - Párrafos válidos etiquetados.
    - Número de bloques considerados.
    - Número de bloques descartados por las reglas de limpieza.
    """

    parrafos_validos = []
    total_bloques = 0
    bloques_descartados = 0
    omitir_seccion = False

    for linea in texto_articulo.splitlines():

        p = linea.strip()

        if not p:
            continue

        total_bloques += 1
        titulo_seccion = None

        # Identificamos encabezados para excluir los apartados no informativos.
        encabezado = re.match(
            r"^(={2,})\s*(.*?)\s*\1\s*$",
            p
        )

        if encabezado:
            titulo_seccion = encabezado.group(2).strip().lower()

        elif p.lower().strip(" .:") in SECCIONES_RUIDOSAS:
            # Algunas secciones llegan sin los signos de encabezado.
            titulo_seccion = p.lower().strip(" .:")

        if titulo_seccion is not None:

            omitir_seccion = any(
                nombre in titulo_seccion
                for nombre in SECCIONES_RUIDOSAS
            )

            bloques_descartados += 1
            continue

        if omitir_seccion:
            bloques_descartados += 1
            continue

        # Eliminar referencias, URLs y posibles etiquetas HTML.
        p = re.sub(r"\[\d+\]", "", p)
        p = re.sub(r"https?://\S+", "", p)
        p = re.sub(r"<[^>]+>", " ", p)

        # El CSV utiliza ; como separador.
        p = p.replace(";", ",")

        p = re.sub(r"\s+", " ", p).strip()

        # Filtro por longitud.
        if len(p) < LONGITUD_MINIMA_PARRAFO:
            bloques_descartados += 1
            continue

        parrafos_validos.append((p, clase))

    return parrafos_validos, total_bloques, bloques_descartados


### 1.4. Generación y comprobación del CSV

Cada párrafo conserva la categoría del artículo del que procede. Al combinar los textos, eliminamos duplicados y excluimos los fragmentos idénticos que hayan recibido etiquetas distintas. El resultado se guarda en UTF-8 con el formato `text;class`.

In [4]:
# Validación de columnas, valores y duplicados
def validar_dataset(df):
    """Valida el contrato text;class y limpia filas problemáticas."""

    columnas = ["text", "class"]

    faltantes = [
        col for col in columnas
        if col not in df.columns
    ]

    if faltantes:
        raise ValueError(
            f"Faltan columnas obligatorias: {faltantes}"
        )

    df = df[columnas].copy()
    df = df.dropna(subset=columnas)

    df["text"] = df["text"].astype(str).str.strip()
    df["class"] = df["class"].astype(str).str.strip()

    df = df[
        (df["text"] != "") &
        (df["class"] != "")
    ]

    df["text"] = df["text"].str.replace(
        ";", ",", regex=False
    )

    # Quitamos repeticiones exactas dentro de cada clase.
    df = df.drop_duplicates(subset=["text", "class"])

    # Un mismo párrafo no debería aparecer con dos clases distintas.
    clases_por_texto = df.groupby("text")["class"].nunique()

    textos_ambiguos = clases_por_texto[
        clases_por_texto > 1
    ].index

    if len(textos_ambiguos) > 0:
        print(
            "Textos con etiquetas contradictorias descartados:",
            len(textos_ambiguos)
        )

        df = df[
            ~df["text"].isin(textos_ambiguos)
        ]

    return df.reset_index(drop=True)


In [5]:
# Construcción del dataset y resumen de resultados
def generar_dataset():
    """Descarga, limpia y construye el dataset de las tres categorías."""

    filas = []
    bloques_totales = 0
    bloques_descartados = 0

    articulos_con_texto = {
        clase: 0 for clase in CATEGORIAS
    }

    for clase, articulos in CATEGORIAS.items():

        print(
            f"\nDescargando artículos de la categoría: {clase}"
        )

        parrafos_clase = 0

        for articulo in articulos:

            # Cada título se consulta por separado.
            texto = obtener_texto_wikipedia(articulo)

            if texto:

                articulos_con_texto[clase] += 1

                parrafos, total, descartados = (
                    limpiar_y_filtrar_parrafos(
                        texto, clase
                    )
                )

                filas.extend(parrafos)

                bloques_totales += total
                bloques_descartados += descartados

                parrafos_clase += len(parrafos)

            # Espaciar las peticiones a Wikipedia.
            time.sleep(PAUSA_ENTRE_ARTICULOS)

        print(
            f"Artículos con texto: "
            f"{articulos_con_texto[clase]} de {len(articulos)}"
        )

        print(
            f"Párrafos válidos en {clase}: "
            f"{parrafos_clase}"
        )

    df_resultado = validar_dataset(
        pd.DataFrame(filas, columns=["text", "class"])
    )

    conteos = df_resultado["class"].value_counts()

    print("\n--- MÉTRICAS DE LIMPIEZA ---")
    print("Bloques no vacíos considerados:", bloques_totales)
    print("Bloques descartados por las reglas:", bloques_descartados)

    tasa = (
        bloques_descartados / bloques_totales * 100
        if bloques_totales else 0
    )

    print(f"Tasa de bloques descartados en la limpieza: {tasa:.2f}%")
    print("La tasa refleja bloques descartados, no la calidad de las etiquetas.")

    print("\n--- PÁRRAFOS POR CLASE ---")

    for clase in CATEGORIAS:
        print(
            f"{clase}: {int(conteos.get(clase, 0))}"
        )

    clases_vacias = [
        clase for clase in CATEGORIAS
        if conteos.get(clase, 0) == 0
    ]

    clases_insuficientes = [
        clase for clase in CATEGORIAS
        if conteos.get(clase, 0) < MIN_PARRAFOS_POR_CLASE
    ]

    if df_resultado.empty or clases_vacias:
        raise ValueError(
            "Dataset incompleto. Clases sin texto: "
            f"{clases_vacias}. No se guardará el CSV."
        )

    if clases_insuficientes:
        raise ValueError(
            f"Hay clases con menos de {MIN_PARRAFOS_POR_CLASE} "
            f"párrafos: {clases_insuficientes}. "
            "Amplía los artículos semilla o comprueba la descarga. "
            "No se ha guardado el CSV."
        )

    # El CSV se guarda solo si las tres clases tienen suficientes datos.
    df_resultado.to_csv(
        "esperanto_scienco_dataset.csv",
        sep=";",
        index=False,
        encoding="utf-8"
    )

    print("\nDataset guardado: esperanto_scienco_dataset.csv")
    print("Filas totales:", len(df_resultado))
    print(df_resultado.head())

    print(
        "\nNota: el enunciado recomienda más de 300 "
        "párrafos por clase."
    )

    if any(
        conteos.get(clase, 0) < 300
        for clase in CATEGORIAS
    ):
        print(
            "Aviso: alguna clase sigue por debajo "
            "de la recomendación de 300 párrafos."
        )

    return df_resultado


In [6]:
# Generación nueva o lectura del CSV guardado.
if DESCARGAR_DESDE_WIKIPEDIA:

    df = generar_dataset()

else:

    try:
        df = pd.read_csv(
            "esperanto_scienco_dataset.csv",
            sep=";",
            encoding="utf-8"
        )

        print("Dataset cargado desde el archivo local.")

    except FileNotFoundError:

        print(
            "No hay CSV local; se intentará cargar desde GitHub."
        )

        try:
            df = pd.read_csv(
                GITHUB_CSV_URL,
                sep=";",
                encoding="utf-8"
            )

        except Exception as error:
            raise RuntimeError(
                "No se pudo cargar el CSV local ni desde GitHub. "
                "Genera el dataset o publica el CSV en el repositorio."
            ) from error

    df = validar_dataset(df)

    clases_encontradas = set(df["class"].unique())
    clases_esperadas = set(CATEGORIAS.keys())

    if not clases_esperadas.issubset(clases_encontradas):
        raise ValueError(
            "El CSV no contiene las tres clases. "
            f"Clases encontradas: {clases_encontradas}. "
            "No continúes con este dataset incompleto."
        )

    print("Filas cargadas:", len(df))
    print(df["class"].value_counts())



Descargando artículos de la categoría: biologio
Artículos con texto: 13 de 13
Párrafos válidos en biologio: 406

Descargando artículos de la categoría: kemio
Artículos con texto: 10 de 10
Párrafos válidos en kemio: 435

Descargando artículos de la categoría: fiziko
Artículos con texto: 13 de 13
Párrafos válidos en fiziko: 353

--- MÉTRICAS DE LIMPIEZA ---
Bloques no vacíos considerados: 2659
Bloques descartados por las reglas: 1465
Tasa de bloques descartados en la limpieza: 55.10%
La tasa refleja bloques descartados, no la calidad de las etiquetas.

--- PÁRRAFOS POR CLASE ---
biologio: 406
kemio: 431
fiziko: 353

Dataset guardado: esperanto_scienco_dataset.csv
Filas totales: 1190
                                                text     class
0  Biologio estas la scienco pri la fenomenoj de ...  biologio
1  Tiu scienco studas la karakterojn, nome fizika...  biologio
2  Biologio ampleksas larĝan spektron de sciencaj...  biologio
3  Sub-disciplinoj de biologio estas difinitaj la...  bio

### 1.5. Calidad de los datos

Revisamos el formato del CSV, comprobamos los recuentos y mostramos una pequeña muestra aleatoria por categoría.

**Limitación:** el párrafo hereda la etiqueta de su artículo. Algunos textos pueden ser demasiado generales o tratar cuestiones compartidas entre varias ciencias. La muestra siguiente sirve para detectar esos casos manualmente; por sí sola no proporciona un porcentaje de error en las etiquetas.

In [7]:
# Verificación del contrato y muestra para revisión
# Condiciones de formato que debe cumplir el CSV.
assert list(df.columns) == ["text", "class"]
assert set(df["class"]) == set(CATEGORIAS)
assert not df[["text", "class"]].isna().any().any()
assert not df.duplicated(subset=["text", "class"]).any()
assert not df["text"].str.contains(";", regex=False).any()

conteos_finales = df["class"].value_counts().reindex(CATEGORIAS.keys())
print("Distribución final del dataset:")
print(conteos_finales.to_string())

# Comprobamos la recomendación de más de 300 párrafos por clase.
if (conteos_finales <= 300).any():
    print("Aviso: ampliar las clases con 300 o menos párrafos.")
else:
    print("Las tres categorías superan los 300 párrafos recomendados.")

# Elegimos la misma muestra en cada ejecución para facilitar su revisión.
print("\nMuestra para inspección manual (3 párrafos por clase):")
muestra_revision = (
    df.groupby("class", group_keys=False)
      .sample(n=min(3, int(conteos_finales.min())), random_state=42)
      .reset_index(drop=True)
)
display(muestra_revision)
print("La muestra requiere revisión manual para valorar la etiqueta.")


Distribución final del dataset:
class
biologio    406
kemio       431
fiziko      353
Las tres categorías superan los 300 párrafos recomendados.

Muestra para inspección manual (3 párrafos por clase):


,text,class
0,"La fotosintezo havas du ŝtupojn, kiuj estas la...",biologio
1,"Science Aid: Ecology, U.K. High School (GCSE, ...",biologio
2,La genetika inĝenierado aŭ gen-tekniko estas t...,biologio
3,Ĥemia termodinamiko estas la studo de la inter...,fiziko
4,"Je supraĵnivela analizo, la vorto konsistas el...",fiziko
5,La ideo ke materio kunmetiĝas el elementaj par...,fiziko
6,Tiu nova modelo en kiu la elektronoj de atomo ...,kemio
7,"Acidoj estas korodemaj. Generale, acidaj brula...",kemio
8,"Theodor Svedberg, 1884–1971, koloida ĥemio, no...",kemio


La muestra requiere revisión manual para valorar la etiqueta.


## 2. Procesamiento del lenguaje natural (NLP)

Implementamos un procesador propio, sin emplear librerías de NLP preexistentes. Usamos reglas sencillas del Esperanto para dividir el texto, estimar los lemas y asignar categorías gramaticales.

La interfaz se inspira en spaCy: cada palabra se representa con un objeto `Token` y el texto completo con un objeto `Doc`. **Las etiquetas y los lemas son aproximados**, ya que no utilizamos un diccionario lingüístico completo.

### 2.1. Stopwords y palabras funcionales

Las *stopwords* son palabras frecuentes que normalmente aportan poca información temática. Además, guardamos varias palabras funcionales junto con su lema y etiqueta gramatical, para tratarlas antes de aplicar las reglas generales.

In [8]:
# Lista de stopwords y palabras con análisis predefinido
ESPERANTO_STOPWORDS = {
    "la", "kaj", "de", "en", "al", "estas", "esti",
    "estis", "estos", "por", "kun", "per", "sur",
    "pri", "sed", "ĉar", "ke", "ne", "kiel", "tiu",
    "tiuj", "tiun", "tiujn", "kiu", "kiuj", "kiun",
    "kiujn", "mi", "vi", "li", "ŝi", "ĝi", "ili",
    "ni", "oni", "sia", "sian", "siaj", "siajn",
    "ankaŭ", "nur", "jam", "pli", "plej", "tre",
    "tro", "ĉe", "post", "dum"
}

# Algunas palabras frecuentes necesitan una etiqueta conocida,
# porque no se pueden clasificar solo por su terminación.

PALABRAS_FUNCION = {
    "la": ("la", "DET"),

    "kaj": ("kaj", "CCONJ"),
    "sed": ("sed", "CCONJ"),
    "ĉar": ("ĉar", "SCONJ"),
    "ke": ("ke", "SCONJ"),

    "de": ("de", "ADP"),
    "en": ("en", "ADP"),
    "al": ("al", "ADP"),
    "por": ("por", "ADP"),
    "kun": ("kun", "ADP"),
    "per": ("per", "ADP"),
    "sur": ("sur", "ADP"),
    "pri": ("pri", "ADP"),
    "ĉe": ("ĉe", "ADP"),
    "post": ("post", "ADP"),
    "dum": ("dum", "ADP"),

    "ne": ("ne", "PART"),

    "kiel": ("kiel", "ADV"),
    "jam": ("jam", "ADV"),
    "ankaŭ": ("ankaŭ", "ADV"),
    "nur": ("nur", "ADV"),

    "mi": ("mi", "PRON"),
    "vi": ("vi", "PRON"),
    "li": ("li", "PRON"),
    "ŝi": ("ŝi", "PRON"),
    "ĝi": ("ĝi", "PRON"),
    "ili": ("ili", "PRON"),
    "ni": ("ni", "PRON"),
    "oni": ("oni", "PRON"),

    "tiu": ("tiu", "DET"),
    "tiuj": ("tiu", "DET"),
    "tiun": ("tiu", "DET"),
    "tiujn": ("tiu", "DET"),

    "kiu": ("kiu", "PRON"),
    "kiuj": ("kiu", "PRON"),
    "kiun": ("kiu", "PRON"),
    "kiujn": ("kiu", "PRON"),
}

print("Stop-words y palabras funcionales cargadas.")


Stop-words y palabras funcionales cargadas.


### 2.2. Clases `Token` y `Doc`

`Token` almacena el texto original, el lema, la etiqueta gramatical y dos indicadores: si es *stopword* y si es puntuación. `Doc` reúne los tokens, permite recorrerlos y reconstruye un texto lematizado sin puntuación o stopwords, cuando se desee.

In [9]:
# Estructura de los resultados del análisis
class Token:
    """Representa un token con su lema y propiedades gramaticales."""

    def __init__(self, text, lemma, pos, is_stop, is_punct):
        self.text = text
        self.lemma_ = lemma
        self.pos_ = pos
        self.is_stop = is_stop
        self.is_punct = is_punct

    def __repr__(self):
        return (
            f"Token(text={self.text!r}, lemma={self.lemma_!r}, "
            f"pos={self.pos_!r}, stop={self.is_stop}, "
            f"punct={self.is_punct})"
        )


class Doc:
    """Agrupa los tokens y permite recorrerlos o recuperar sus lemas."""

    def __init__(self, tokens, original_text):
        self._tokens = tokens
        self._text = original_text

    def __iter__(self):
        return iter(self._tokens)

    def __len__(self):
        return len(self._tokens)

    def __getitem__(self, index):
        return self._tokens[index]

    @property
    def text(self):
        return self._text

    def lemmatized_text(
        self,
        remove_stopwords=True,
        remove_punct=True
    ):
        lemmas = []

        for token in self._tokens:

            if remove_punct and token.is_punct:
                continue

            if remove_stopwords and token.is_stop:
                continue

            lemmas.append(token.lemma_)

        return " ".join(lemmas)


### 2.3. Reglas de lematización y etiquetado

Las terminaciones ayudan a reconocer formas como `ĉeloj → ĉelo` (plural) y `legis → legi` (verbo en pasado). Para palabras no reconocidas, el sistema utiliza la etiqueta `X`.

Este enfoque es fácil de interpretar, aunque tiene limitaciones con palabras compuestas, formas menos frecuentes y excepciones léxicas.

In [10]:
# Procesador lingüístico basado en reglas
class EsperantoNLP:
    """Pipeline aproximado basado en reglas para texto en Esperanto."""

    def __init__(self, stopwords=None):
        self.stopwords = (
            set(ESPERANTO_STOPWORDS)
            if stopwords is None
            else set(stopwords)
        )

    def _morphological_analysis(self, word):
        w = word.lower()

        # Las palabras funcionales se consultan antes de aplicar sufijos.
        if w in PALABRAS_FUNCION:
            return PALABRAS_FUNCION[w]

        # Terminaciones habituales de los verbos en Esperanto.
        for ending in ("as", "is", "os", "us"):

            if w.endswith(ending) and len(w) > len(ending):
                return w[:-len(ending)] + "i", "VERB"

        if w.endswith("u") and len(w) > 2:
            return w[:-1] + "i", "VERB"

        if w.endswith("i") and len(w) > 2:
            return w, "VERB"

        # Quitamos marcas de plural (-j) y acusativo (-n) cuando procede.
        base = w

        if (
            len(w) > 3
            and w.endswith("jn")
            and w[:-2].endswith(("o", "a"))
        ):
            base = w[:-2]

        elif (
            len(w) > 2
            and w.endswith("n")
            and w[:-1].endswith(("o", "a"))
        ):
            base = w[:-1]

        elif (
            len(w) > 2
            and w.endswith("j")
            and w[:-1].endswith(("o", "a"))
        ):
            base = w[:-1]

        # La última letra ayuda a reconocer sustantivos, adjetivos y adverbios.
        if base.endswith("o") and len(base) > 1:
            return base, "NOUN"

        if base.endswith("a") and len(base) > 1:
            return base, "ADJ"

        if base.endswith("e") and len(base) > 1:
            return base, "ADV"

        return w, "X"

    def __call__(self, text):

        if not isinstance(text, str):
            raise TypeError(
                "EsperantoNLP espera una cadena de texto."
            )

        # La expresión regular separa palabras y signos de puntuación.
        raw_tokens = re.findall(
            r"\w+|[^\w\s]",
            text,
            flags=re.UNICODE
        )

        doc_tokens = []

        for raw in raw_tokens:

            is_punct = bool(
                re.fullmatch(
                    r"[^\w\s]",
                    raw,
                    flags=re.UNICODE
                )
            )

            lower_raw = raw.lower()

            if is_punct:
                lemma, pos = raw, "PUNCT"
            else:
                lemma, pos = self._morphological_analysis(raw)

            token = Token(
                text=raw,
                lemma=lemma,
                pos=pos,
                is_stop=lower_raw in self.stopwords,
                is_punct=is_punct
            )

            doc_tokens.append(token)

        return Doc(doc_tokens, text)


nlp = EsperantoNLP()

print("Modelo EsperantoNLP inicializado.")


Modelo EsperantoNLP inicializado.


### 2.4. Ejemplo y pruebas básicas

Probamos una frase corta para comprobar qué devuelve el procesador. Después usamos `assert` con palabras y casos concretos para verificar que los cambios no alteran el comportamiento esperado.

In [11]:
# Ejemplo de salida de Token y Doc
ejemplo = "La grandaj ĉeloj dividiĝas rapide en la laboratorio."

doc = nlp(ejemplo)

print("TEXTO ORIGINAL:")
print(doc.text)

print("\nTOKENS:")
print(
    f"{'TEXTO':<15} | {'LEMA':<15} | "
    f"{'POS':<8} | {'STOPWORD':<9} | PUNT"
)
print("-" * 65)

for token in doc:
    print(
        f"{token.text:<15} | {token.lemma_:<15} | "
        f"{token.pos_:<8} | {str(token.is_stop):<9} | "
        f"{token.is_punct}"
    )

print("\nTexto lematizado sin stop-words ni puntuación:")

print(
    doc.lemmatized_text(
        remove_stopwords=True,
        remove_punct=True
    )
)


TEXTO ORIGINAL:
La grandaj ĉeloj dividiĝas rapide en la laboratorio.

TOKENS:
TEXTO           | LEMA            | POS      | STOPWORD  | PUNT
-----------------------------------------------------------------
La              | la              | DET      | True      | False
grandaj         | granda          | ADJ      | False     | False
ĉeloj           | ĉelo            | NOUN     | False     | False
dividiĝas       | dividiĝi        | VERB     | False     | False
rapide          | rapide          | ADV      | False     | False
en              | en              | ADP      | True      | False
la              | la              | DET      | True      | False
laboratorio     | laboratorio     | NOUN     | False     | False
.               | .               | PUNCT    | False     | True

Texto lematizado sin stop-words ni puntuación:
granda ĉelo dividiĝi rapide laboratorio


In [12]:
# Comprobaciones automatizadas de las reglas implementadas
pruebas = {
    "kaj": ("kaj", "CCONJ", True),
    "kun": ("kun", "ADP", True),
    "en": ("en", "ADP", True),
    "ĉeloj": ("ĉelo", "NOUN", False),
    "ĉelojn": ("ĉelo", "NOUN", False),
    "grandajn": ("granda", "ADJ", False),
    "domojn": ("domo", "NOUN", False),
    "legis": ("legi", "VERB", False),
    "iras": ("iri", "VERB", False),
    "estas": ("esti", "VERB", True),
    "rapide": ("rapide", "ADV", False),
}

for palabra, esperado in pruebas.items():

    token = nlp(palabra)[0]

    obtenido = (
        token.lemma_,
        token.pos_,
        token.is_stop
    )

    assert obtenido == esperado, (
        f"Error con {palabra!r}: "
        f"esperado {esperado}, obtenido {obtenido}"
    )

# Conservamos el texto original y admitimos cadenas vacías.
assert nlp(ejemplo).text == ejemplo
assert len(nlp("")) == 0

# Comprobamos la puntuación y el lema de una forma simple.
assert nlp("Saluton.").lemmatized_text() == "saluto"

# Se mantienen caracteres propios del Esperanto como ĉ y ĝ.
assert [t.text for t in nlp("Ĉeloj, ĝardenoj!")] == [
    "Ĉeloj", ",", "ĝardenoj", "!"
]
assert nlp("Saluton, mondo!")[1].is_punct

# La entrada debe ser una cadena de texto.
try:
    nlp(42)
    raise AssertionError("Debería rechazar un número.")
except TypeError:
    pass

print("\nTodas las pruebas básicas del modelo NLP han pasado.")



Todas las pruebas básicas del modelo NLP han pasado.


## 3. Estado de la entrega intermedia

Las fases 1 y 2 quedan preparadas para su comprobación: dataset etiquetado, limpieza, funciones de NLP y pruebas básicas. Antes de entregar, ejecutamos el notebook completo en un entorno limpio y revisamos los avisos de descarga y la muestra de párrafos.

**Pendiente para la entrega final:** vectorización del corpus, entrenamiento, evaluación del clasificador y clasificación de páginas web reales.